# FloodVision GB10

Analyze flood images with **Qwen3-VL-8B**, generate a visual flood assessment, a **0–100 hazard indicator**, and an estimated walking time **only for a separately verified dry route**.

**Safety:** The hazard indicator is not an injury probability. Images cannot reliably measure water depth, current, or evacuation safety. Never enter or drive through floodwater.

### 1. Install packages

In [ ]:
%pip install -q openai pillow pandas matplotlib

### 2. Set up the model and images

Set `MODE` to `"single"` for separate reports or `"before_after"` to compare exactly two photos. Add your image file paths to `IMAGE_PATHS`.

In [ ]:
import base64
import io
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image, ImageOps
from openai import OpenAI

IP_ADDRESS = "http://10.246.100.230/v1"
MODEL = "qwen3-vl-8b-instruct"
client = OpenAI(base_url=IP_ADDRESS, api_key=os.getenv("GB10_API_KEY", "usaadatathon"), timeout=120)

MODE = "single"
IMAGE_PATHS = []
LOCATION = "Not specified"
VERIFIED_DRY_ROUTE = False
DRY_ROUTE_DISTANCE_M = None

print(f"Model ready: {MODEL}")

### 3. Load images

In [ ]:
def image_to_url(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        image.thumbnail((1280, 1280))
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=85)
    return "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode("utf-8")


for path in IMAGE_PATHS:
    with Image.open(path) as image:
        preview = ImageOps.exif_transpose(image).copy()
        preview.thumbnail((700, 450))
        print(Path(path).name)
        display(preview)

### 4. Analyze images with Qwen3-VL

In [ ]:
PROMPT = """
You are analyzing visible flood conditions in photographs. Respond in English.
Use only observable evidence. Do not guess water depth, current speed, distances, injury probability, or route safety.
Do not confuse ordinary rivers, lakes, or wet pavement with flooding.
If two images are provided, the first is BEFORE and the second is AFTER. Evaluate the AFTER image.
Use "yes", "no", or "uncertain" for each yes/no field. Flood extent must be "none", "localized", "widespread", or "uncertain".
A visible dry-looking exit does not prove the route is safe.
Return ONLY a JSON object using this structure:
{
  "flooding_visible": "yes|no|uncertain",
  "flood_extent": "none|localized|widespread|uncertain",
  "road_or_path_flooded": "yes|no|uncertain",
  "moving_water_cues": "yes|no|uncertain",
  "visible_debris": "yes|no|uncertain",
  "people_exposed": "yes|no|uncertain",
  "vehicles_exposed": "yes|no|uncertain",
  "safe_dry_exit_visible": "yes|no|uncertain",
  "observations": ["2 to 4 visible facts"],
  "uncertainties": ["2 to 4 things the image cannot verify"],
  "summary": "Two-sentence summary without invented measurements",
  "change_from_before": "Visible changes if there are two photos; otherwise empty string"
}
""".strip()

FIELDS = [
    "flooding_visible", "road_or_path_flooded", "moving_water_cues",
    "visible_debris", "people_exposed", "vehicles_exposed", "safe_dry_exit_visible"
]


def parse_response(text):
    text = text.strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        if start < 0:
            raise ValueError(f"No JSON found in the model response: {text[:200]}")
        data, _ = json.JSONDecoder().raw_decode(text[start:])

    if not isinstance(data, dict):
        raise ValueError("Expected a JSON object")
    for field in FIELDS:
        value = str(data.get(field, "uncertain")).lower().strip()
        data[field] = value if value in {"yes", "no", "uncertain"} else "uncertain"
    extent = str(data.get("flood_extent", "uncertain")).lower().strip()
    data["flood_extent"] = extent if extent in {"none", "localized", "widespread", "uncertain"} else "uncertain"
    for field in ("observations", "uncertainties"):
        value = data.get(field, [])
        data[field] = [str(item) for item in value[:6] if isinstance(item, str)] if isinstance(value, list) else []
    for field in ("summary", "change_from_before"):
        data[field] = str(data.get(field, ""))
    if data["flooding_visible"] == "no" and (
        data["road_or_path_flooded"] == "yes" or data["moving_water_cues"] == "yes"
    ):
        data["flooding_visible"] = "uncertain"
        data["uncertainties"].append("The observations conflict and need review.")
    return data


def analyze_images(paths):
    content = [{"type": "text", "text": PROMPT}]
    for i, path in enumerate(paths):
        if len(paths) == 2:
            content.append({"type": "text", "text": "BEFORE" if i == 0 else "AFTER"})
        content.append({"type": "image_url", "image_url": {"url": image_to_url(path)}})

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Describe only visible evidence. Return valid JSON. Never certify safety."},
            {"role": "user", "content": content},
        ],
        temperature=0.001,
        max_tokens=1200,
    )
    return parse_response(response.choices[0].message.content or "")

### 5. Calculate the hazard indicator and dry-route travel time

In [ ]:
HAZARD_WEIGHTS = [
    ("flood_extent", "widespread", 8, "Widespread water"),
    ("road_or_path_flooded", "yes", 15, "Flooded road or path"),
    ("moving_water_cues", "yes", 18, "Visible signs of moving water"),
    ("visible_debris", "yes", 8, "Debris or obstacles"),
    ("people_exposed", "yes", 8, "People in the affected area"),
    ("vehicles_exposed", "yes", 7, "Vehicles in the affected area"),
    ("safe_dry_exit_visible", "no", 12, "No dry-looking exit visible"),
]


def hazard_indicator(data):
    if data["flooding_visible"] != "yes":
        return {"score": None, "level": "Not assessable", "reasons": ["Flooding is not confirmed in the image"]}
    score = 50
    reasons = ["Visible evidence of flooding"]
    for field, expected, points, reason in HAZARD_WEIGHTS:
        if data.get(field) == expected:
            score += points
            reasons.append(f"{reason} (+{points})")
    score = min(score, 100)
    return {"score": score, "level": "Critical" if score >= 80 else "High", "reasons": reasons}


def dry_route_time(distance_m, verified):
    if not verified or distance_m is None:
        return None
    if isinstance(distance_m, bool) or not isinstance(distance_m, (int, float)) or not 0 < distance_m <= 100_000:
        raise ValueError("Enter a valid dry-route distance in meters")
    return {
        "distance_m": distance_m,
        "min_minutes": round(distance_m / 1.2 / 60, 1),
        "max_minutes": round(distance_m / 0.6 / 60, 1),
    }


def build_report(data, paths):
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "location": LOCATION,
        "images": [str(path) for path in paths],
        "model": MODEL,
        "analysis": data,
        "visual_hazard": hazard_indicator(data),
        "dry_route_time": dry_route_time(DRY_ROUTE_DISTANCE_M, VERIFIED_DRY_ROUTE),
    }

### 6. Generate reports

In [ ]:
reports = []

if not IMAGE_PATHS:
    print("Add image paths in Section 2 and run this cell again.")
elif MODE == "before_after":
    if len(IMAGE_PATHS) != 2:
        raise ValueError("Before/after mode requires exactly two images")
    reports.append(build_report(analyze_images(IMAGE_PATHS), IMAGE_PATHS))
elif MODE == "single":
    for path in IMAGE_PATHS:
        reports.append(build_report(analyze_images([path]), [path]))
else:
    raise ValueError("MODE must be 'single' or 'before_after'")

print(f"Reports generated: {len(reports)}")

In [ ]:
def report_markdown(report):
    analysis = report["analysis"]
    hazard = report["visual_hazard"]
    travel = report["dry_route_time"]
    score = f"{hazard['score']}/100 points ({hazard['level']})" if hazard["score"] is not None else "Not assessable"
    time_text = (
        f"{travel['min_minutes']}–{travel['max_minutes']} minutes for {travel['distance_m']} m "
        "on an independently verified dry route (illustrative only)"
        if travel else "Not available: dry-route distance and independent verification required"
    )
    lines = [
        "# FloodVision GB10 — Image Report", "",
        f"**Location:** {report['location']}",
        f"**Images:** {', '.join(report['images'])}",
        f"**Summary:** {analysis['summary']}", "",
        f"**Flood visible:** {analysis['flooding_visible']}",
        f"**Flood extent:** {analysis['flood_extent']}",
        f"**Flooded road/path:** {analysis['road_or_path_flooded']}",
        f"**Signs of moving water:** {analysis['moving_water_cues']}",
        f"**Visual hazard indicator:** {score}",
        f"**Dry-route walking time:** {time_text}", "",
        "## Visual evidence",
        *[f"- {item}" for item in analysis["observations"]], "",
        "## Hazard indicators",
        *[f"- {item}" for item in hazard["reasons"]], "",
        "## Uncertainties",
        *[f"- {item}" for item in analysis["uncertainties"]],
    ]
    if analysis["change_from_before"]:
        lines.extend(["", "## Before and after", analysis["change_from_before"]])
    lines.extend([
        "", "**Safety:** This experimental visual hazard indicator is not an injury probability or a validated safety rating. "
        "A photo cannot verify the depth, current, or whether a route is safe. Never cross floodwater."
    ])
    return "\n".join(lines)


for report in reports:
    display(Markdown(report_markdown(report)))

### 7. Save reports and view scores

In [ ]:
output_dir = Path("floodvision_reports")
if reports:
    output_dir.mkdir(exist_ok=True)
    for i, report in enumerate(reports, 1):
        stem = f"flood_report_{i:02d}"
        (output_dir / f"{stem}.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
        (output_dir / f"{stem}.md").write_text(report_markdown(report), encoding="utf-8")
    print(f"Reports saved in {output_dir}/")

    table = pd.DataFrame([
        {
            "Image": ", ".join(report["images"]),
            "Flood extent": report["analysis"]["flood_extent"],
            "Visual hazard (0–100 points)": report["visual_hazard"]["score"],
        }
        for report in reports
    ])
    display(table)
    valid = table.dropna(subset=["Visual hazard (0–100 points)"])
    if not valid.empty:
        ax = valid.plot.bar(x="Image", y="Visual hazard (0–100 points)", legend=False, figsize=(9, 4))
        ax.set_ylim(0, 100)
        ax.set_ylabel("Heuristic points (not injury probability)")
        plt.tight_layout()
        plt.show()